### Task 9: 

Create an interactive model of the virtual image of an object in a convex spherical mirror.

Have not figured out how to run it in GitHub, and so - for the purposes of the submission video - this task was run in Thonny.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Arc
import numpy as np
from numpy import tan,atan
import os
from PIL import Image

import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
    

def load_img(image_path): # loads image file and converts to numpy array
    try:
        img = Image.open(image_path) 
        return np.array(img)
    except Exception as e:
        print(f"Error loading image: {e}")
        return None
    
def open_img():
    image_path = "stinkbug.png" 
    '''replace this image file if needed!'''
    if not os.path.exists(image_path): # error handling
        print(f"'{image_path}' not found")
        raise SystemExit
    else:
        img_path = load_img(image_path)
        if img_path is None: # more error handling
            print('uhoh')
            raise SystemExit
        print(f"{image_path} successfully loaded")
        return img_path

orig_img = open_img()

def mirror(X, Y, r): # calc transformation w/ circular mirror equation
    m = 2 * Y * (r**2-Y**2)**0.5 / (r**2 - 2*Y**2) # achieved by rearranging and using tan(2*theta) rule
    xx = ( Y - m * ((r**2-Y**2)**0.5))/(Y/X - m)
    mask = np.isnan(xx) # mask for when Y/X - m == 0, pcolormesh cannot take NaN
    xx[mask] = np.interp(np.flatnonzero(mask), 
                      np.flatnonzero(~mask), 
                      xx[~mask])
    yy = Y*xx/X
    return xx, yy

''''''
# scaling image up/down to fit graph
height, width = orig_img.shape[:2]
aspect_ratio = height / width
x0, y0 = 10 , 0    # object centre
obj_w = 5       # width of object (scales image)
# maximum and minimum x and y coords
x_min = x0 - obj_w / 2
x_max = x0 + obj_w / 2
y_min = y0 - obj_w / 2 * aspect_ratio
y_max = y0 + obj_w / 2 * aspect_ratio
# meshgrid
x = np.linspace(0, obj_w, width) - obj_w/2 + x0
y = -np.linspace(0, obj_w*aspect_ratio, height) + obj_w*aspect_ratio/2 + y0
X, Y = np.meshgrid(x,y)
# figure
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
# mirror and focal point plot
mirror_img = Arc((0, 0), width = 10, height = 10, theta1 =-90, theta2 = 90,
                    edgecolor='b', fc = 'None', lw=0.5, label = 'Convex Mirror')
ax.add_patch(mirror_img)
plt.plot(0,0, marker='*', color='red', markersize=3, zorder = 5, label='Mirror Focal Point')
# original/real image
img_show = ax.pcolormesh(X, Y, orig_img, shading='auto', zorder=3)
#transformed/virtual image
r = 5 # radius of mirror
xx,yy = mirror(X, Y, r)
virtual_show = ax.pcolormesh(xx, yy, orig_img, shading='auto', zorder=3)
# axis position, limits and labels
ax.spines['left'].set_position(('outward', 0.8)) # position of y-axis
ax.spines['bottom'].set_position(('outward', 0.8)) # position of x-axis
ax.set_xlim(-0.2, 15)
ax.set_ylim(-5.2, 5.2)
ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('y', fontsize=12)
# title and labels
ax.set_title('Reflection in Convex Mirror', fontsize=16)
plt.text(15.1,5, 'Use Arrow Keys to Move the Image!', fontsize = 8)
obj_txt = ax.text(10, -2.25, 'Real Object', 
            ha='center', fontsize=10)
virtual_txt = ax.text(3.5, -1.25, 'Virtual Object', 
            ha='center', fontsize=10)
uhoh_txt = ax.text(15.1,4.7, '', color = 'red', fontsize = 8)
# grid
ax.grid(True, which = 'both', alpha=0.5, linestyle='-', zorder = 1)
ax.minorticks_on()
ax.set_aspect('equal', adjustable='box')
# legend
plt.legend(loc='upper right', fontsize=12)
''''''

def update(event): # update image position
    global X, Y, x_min, x_max, y_max, y_min
    uhoh_txt.set_text('')
    obj_txt.set_text('')
    virtual_txt.set_text('')
    # moving up
    if event.key == 'up':
        y_max += 0.2
        if y_max > 4:
            uhoh_txt.set_text('Out of Bounds')
            y_max -= 0.2
        else:
            Y += 0.2
            y_min += 0.2
            move_pic()
    #moving down
    elif event.key == 'down':
        y_min -= 0.2
        if y_min <-4: 
            uhoh_txt.set_text('Out of Bounds')
            y_min += 0.2
        else:
            Y -= 0.2
            y_max -= 0.2
            move_pic()
    #moving left
    elif event.key == 'left':
        x_min -= 0.2
        if  x_min < 5: # 
            uhoh_txt.set_text('About to Hit the Mirror!')
            x_min += 0.2
        else:
            X -= 0.2
            x_max -= 0.2
            move_pic()
    #moving right
    elif event.key == 'right':
        x_max += 0.2
        if x_max >= 15:
            uhoh_txt.set_text('Out of Bounds')
            x_max -= 0.2
        else:
            X += 0.2
            x_min += 0.2
            move_pic()
    fig.canvas.draw_idle()
 
def move_pic(): # actually move the images
    global img_show, virtual_show
    xx,yy = mirror(X, Y, r) # re-calculate virtual img shape
    img_show.remove()
    virtual_show.remove()
    img_show = ax.pcolormesh(X, Y, orig_img, shading='auto', zorder=3)
    virtual_show = ax.pcolormesh(xx, yy, orig_img, shading='auto', zorder=3)
    plt.show()

fig.canvas.mpl_connect('key_press_event', update)
plt.show()